In [ ]:
import sys
import pandas as pd
import os
import dotenv

dotenv.load_dotenv(dotenv_path='/Users/mlchrzan/Library/CloudStorage/OneDrive-Personal/Professional/Data Science/Maryland/bechmark analysis pipeline/benchmark-analysis/.env')

sys.path.append('/Users/mlchrzan/Library/CloudStorage/OneDrive-Personal/Professional/Data Science/Personal Projects/pairadigm')
import pairadigm as pr

In [ ]:
sample = pd.read_csv('Data/annotation_sample.csv')
sample

In [ ]:
# Read a txt file with prompts
with open('Data/cgcot_prompts/valence.txt', 'r') as file:
    val_prompts = file.read().splitlines()

with open('Data/cgcot_prompts/arousal.txt', 'r') as file:
    arousal_prompts = file.read().splitlines()

with open('Data/cgcot_prompts/dominance.txt', 'r') as file:
    dom_prompts = file.read().splitlines()

In [ ]:
p_val = pr.Pairadigm(
    sample, 
    item_id_name='id',
    text_name='text',
    cgcot_prompts=val_prompts,
    model_name='gpt-4.1',
    api_key=os.getenv('OPENAI_API_KEY'),
    target_concept='valence'
)

p_aro = pr.Pairadigm(
    sample, 
    item_id_name='id',
    text_name='text',
    cgcot_prompts=arousal_prompts,
    model_name='gpt-4.1',
    api_key=os.getenv('OPENAI_API_KEY'),
    target_concept='arousal'
)

p_dom = pr.Pairadigm(
    sample, 
    item_id_name='id',
    text_name='text',
    cgcot_prompts=dom_prompts,
    model_name='gpt-4.1',
    api_key=os.getenv('OPENAI_API_KEY'),
    target_concept='dominance'
)

In [ ]:
pairs = p_val.pairwise_df[['item1', 'item2']].copy()

pairs_for_humans = pairs.merge(sample[['id', 'text']], left_on='item1', right_on='id').rename(columns={'text': 'text1'}).drop(columns=['id'])
pairs_for_humans = pairs_for_humans.merge(sample[['id', 'text']], left_on='item2', right_on='id').rename(columns={'text': 'text2'}).drop(columns=['id'])

pairs_for_humans.to_excel('Data/human annotations/pairs_for_human_annotation.xlsx', index=False)

## Exploration (ignore this)

In [ ]:
for p in [p_val, p_aro, p_dom]:
    p.generate_breakdowns(rate_limit_per_minute=0)
    p.generate_pairings(breakdowns=True)
    p.generate_pairwise_annotations(max_workers=8)
    p.score_items(normalization_scale="negative-one-to-one")
    

In [ ]:
p_val.scored_df

In [ ]:
all_scores = sample.copy()
for p in [p_val, p_aro, p_dom]:
    new_df = p.scored_df.copy()
    new_df = new_df[['id', 'Bradley_Terry_Score']] if "Bradley_Terry_Score" in new_df.columns else new_df[['id', f'BT_Score_{p.target_concept}']]
    new_df.rename(columns={'Bradley_Terry_Score': f'BT_Score_{p.target_concept}'}, inplace=True)
    all_scores = new_df if all_scores is None else all_scores.merge(new_df, on='id')

all_scores

In [ ]:
all_scores.to_csv('Data/gpt_4-1_annotation_scores.csv', index=False)

In [ ]:
from scipy.stats import pearsonr

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Valence vs BT_Score_valence
corr_v, _ = pearsonr(all_scores['V'], all_scores['BT_Score_valence'])
axes[0].scatter(all_scores['V'], all_scores['BT_Score_valence'], alpha=0.6)
axes[0].set_xlabel('Valence (V)')
axes[0].set_ylabel('BT_Score_valence')
axes[0].set_title(f'Valence vs BT Score\nr = {corr_v:.3f}')
axes[0].grid(True, alpha=0.3)

# Arousal vs BT_Score_arousal
corr_a, _ = pearsonr(all_scores['A'], all_scores['BT_Score_arousal'])
axes[1].scatter(all_scores['A'], all_scores['BT_Score_arousal'], alpha=0.6)
axes[1].set_xlabel('Arousal (A)')
axes[1].set_ylabel('BT_Score_arousal')
axes[1].set_title(f'Arousal vs BT Score\nr = {corr_a:.3f}')
axes[1].grid(True, alpha=0.3)

# Dominance vs BT_Score_dominance
corr_d, _ = pearsonr(all_scores['D'], all_scores['BT_Score_dominance'])
axes[2].scatter(all_scores['D'], all_scores['BT_Score_dominance'], alpha=0.6)
axes[2].set_xlabel('Dominance (D)')
axes[2].set_ylabel('BT_Score_dominance')
axes[2].set_title(f'Dominance vs BT Score\nr = {corr_d:.3f}')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
p_val.save('Data/pairadigm_pickles/pairadigm_valence.pkl')
p_aro.save('Data/pairadigm_pickles/pairadigm_arousal.pkl')
p_dom.save('Data/pairadigm_pickles/pairadigm_dominance.pkl')